# Aula 19 — Feature engineering e seleção de variáveis

Laboratório reproduzível para comparar representações e expor dois vazamentos silenciosos: selecionar features antes da validação e usar informação posterior ao instante de predição.

**Hipóteses registradas antes da execução**

1. seno/cosseno representarão melhor uma relação horária circular que a hora bruta em regressão logística;
2. uma interação quadrática tornará um padrão XOR-like acessível a um modelo linear;
3. seleção supervisionada feita no dataset completo inflará a validação em dados sem sinal;
4. uma variável pós-desfecho parecerá quase perfeita, embora seja indisponível em \(t_0\).

Os dados são sintéticos, a seed é fixa e o teste da primeira investigação permanece lacrado até a escolha da representação.

## Ambiente

Dependências mínimas: Python 3.10, NumPy 1.26, Matplotlib 3.8 e scikit-learn 1.4.

Em um Colab novo, descomente e execute:

```python
# %pip install "numpy>=1.26" "matplotlib>=3.8" "scikit-learn>=1.4"
```

In [ ]:
import platform
import warnings
from itertools import combinations

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import sklearn
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, PolynomialFeatures, StandardScaler

warnings.simplefilter("error")
SEED = 20260908
rng = np.random.default_rng(SEED)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("Matplotlib:", matplotlib.__version__)
print("seed:", SEED)

## 1. Hora bruta versus representação cíclica

A unidade de análise é uma observação independente. O alvo é mais provável perto da meia-noite. Criamos desenvolvimento e teste estratificados antes de comparar representações. A transformação seno/cosseno usa apenas o período de 24 horas definido pelo domínio; não aprende estatística do dataset.

In [ ]:
n = 2_400
hour = rng.integers(0, 24, size=n)
latent_score = 2.8 * np.cos(2 * np.pi * hour / 24) + rng.normal(0, 1.25, n) - 1.15
y_hour = (latent_score > 0).astype(int)

X_dev_h, X_test_h, y_dev_h, y_test_h = train_test_split(
    hour[:, None],
    y_hour,
    test_size=0.25,
    stratify=y_hour,
    random_state=SEED,
)
print("desenvolvimento/teste:", X_dev_h.shape, X_test_h.shape)
print("prevalência:", round(y_hour.mean(), 6))

In [ ]:
def cyclical_hour(X):
    h = np.asarray(X)[:, 0]
    return np.column_stack((
        np.sin(2 * np.pi * h / 24),
        np.cos(2 * np.pi * h / 24),
    ))

raw_hour_pipe = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2_000, random_state=SEED),
)
cyclic_hour_pipe = make_pipeline(
    FunctionTransformer(cyclical_hour, validate=False),
    StandardScaler(),
    LogisticRegression(max_iter=2_000, random_state=SEED),
)

raw_hour_cv = cross_val_score(
    raw_hour_pipe, X_dev_h, y_dev_h, cv=cv, scoring="roc_auc"
)
cyclic_hour_cv = cross_val_score(
    cyclic_hour_pipe, X_dev_h, y_dev_h, cv=cv, scoring="roc_auc"
)
print(f"hora bruta CV: {raw_hour_cv.mean():.6f} ± {raw_hour_cv.std():.6f}")
print(f"cíclica CV:    {cyclic_hour_cv.mean():.6f} ± {cyclic_hour_cv.std():.6f}")

A escolha é feita apenas com desenvolvimento. Agora ajustamos a representação cíclica em todo o desenvolvimento e consultamos o teste uma única vez.

In [ ]:
cyclic_hour_pipe.fit(X_dev_h, y_dev_h)
hour_test_auc = roc_auc_score(
    y_test_h, cyclic_hour_pipe.predict_proba(X_test_h)[:, 1]
)
grid_hour = np.arange(24)[:, None]
grid_prob = cyclic_hour_pipe.predict_proba(grid_hour)[:, 1]

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(grid_hour[:, 0], grid_prob, marker="o")
ax.set(
    xlabel="hora",
    ylabel="probabilidade prevista",
    title="Relação periódica aprendida com seno e cosseno",
    xticks=np.arange(0, 24, 3),
)
ax.grid(alpha=0.25)
plt.show()
print(f"ROC-AUC no teste lacrado: {hour_test_auc:.6f}")

**Leitura:** a hora bruta impõe ordem linear; duas coordenadas cíclicas fecham o relógio. O ganho sustenta esta representação para o processo sintético, não uma regra universal para qualquer timestamp.

## 2. Uma interação que features isoladas escondem

Geramos duas variáveis gaussianas. A chance da classe positiva depende de \(x_1x_2\), não de uma tendência linear isolada. Usamos os mesmos folds para o baseline e para `PolynomialFeatures`.

In [ ]:
n_xor = 1_800
X_xor = rng.normal(size=(n_xor, 2))
xor_logit = 3.4 * (X_xor[:, 0] * X_xor[:, 1])
xor_prob = 1 / (1 + np.exp(-xor_logit))
y_xor = rng.binomial(1, xor_prob)

def polynomial_pipe(degree):
    return make_pipeline(
        PolynomialFeatures(degree=degree, include_bias=False),
        StandardScaler(),
        LogisticRegression(max_iter=2_000, random_state=SEED),
    )

xor_raw = cross_val_score(
    polynomial_pipe(1), X_xor, y_xor, cv=cv, scoring="roc_auc"
)
xor_poly = cross_val_score(
    polynomial_pipe(2), X_xor, y_xor, cv=cv, scoring="roc_auc"
)
f_scores, f_pvalues = f_classif(X_xor, y_xor)
print("scores F isolados:", np.round(f_scores, 6))
print("p-valores isolados:", np.round(f_pvalues, 6))
print(f"linear bruto CV: {xor_raw.mean():.6f} ± {xor_raw.std():.6f}")
print(f"grau 2 CV:       {xor_poly.mean():.6f} ± {xor_poly.std():.6f}")

In [ ]:
poly_check = PolynomialFeatures(degree=2, include_bias=False).fit(X_xor)
names = poly_check.get_feature_names_out(["x1", "x2"])
print("features geradas:", names.tolist())
assert poly_check.n_output_features_ == 5

fig, ax = plt.subplots(figsize=(5, 4))
sample = np.arange(0, n_xor, 8)
ax.scatter(
    X_xor[sample, 0],
    X_xor[sample, 1],
    c=y_xor[sample],
    cmap="coolwarm",
    alpha=0.65,
    s=18,
)
ax.axhline(0, color="black", lw=0.7)
ax.axvline(0, color="black", lw=0.7)
ax.set(xlabel="x1", ylabel="x2", title="Sinal nos quadrantes: interação x1 × x2")
plt.show()

Um filtro univariado tem pouco sinal para ranquear `x1` ou `x2`. Depois da engenharia, `x1 x2` oferece ao classificador linear a coordenada coerente com o mecanismo. O crescimento combinatório de polinômios exige hipótese, regularização e validação.

## 3. Contraprova: seleção antes da CV

Agora \(X\) e \(y\) são ruído independente: não existe sinal generalizável. Com 4.000 features e poucas linhas, haverá correlações aleatórias atraentes.

- **incorreto:** ajustar `SelectKBest` em todas as linhas e depois fazer CV;
- **correto:** colocar `SelectKBest` no pipeline, reajustando-o em cada fold.


In [ ]:
n_noise, p_noise = 260, 4_000
X_noise = rng.normal(size=(n_noise, p_noise))
y_noise = rng.integers(0, 2, size=n_noise)

leaky_selector = SelectKBest(f_classif, k=20).fit(X_noise, y_noise)
X_noise_leaky = leaky_selector.transform(X_noise)
classifier = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2_000, random_state=SEED),
)
leaky_cv = cross_val_score(
    classifier, X_noise_leaky, y_noise, cv=cv, scoring="roc_auc"
)

honest_selection_pipe = make_pipeline(
    SelectKBest(f_classif, k=20),
    StandardScaler(),
    LogisticRegression(max_iter=2_000, random_state=SEED),
)
honest_result = cross_validate(
    honest_selection_pipe,
    X_noise,
    y_noise,
    cv=cv,
    scoring="roc_auc",
    return_estimator=True,
)
honest_cv = honest_result["test_score"]
print(f"seleção antes da CV: {leaky_cv.mean():.6f} ± {leaky_cv.std():.6f}")
print(f"seleção no pipeline: {honest_cv.mean():.6f} ± {honest_cv.std():.6f}")

Também auditamos a estabilidade. Em cada fold, recuperamos os índices escolhidos pelo seletor ajustado somente no respectivo treino e calculamos Jaccard entre todos os pares de listas.

In [ ]:
selected_sets = []
for estimator in honest_result["estimator"]:
    support = estimator.named_steps["selectkbest"].get_support(indices=True)
    selected_sets.append(set(support.tolist()))

jaccards = []
for left, right in combinations(selected_sets, 2):
    jaccards.append(len(left & right) / len(left | right))

print("features selecionadas por fold:", [len(s) for s in selected_sets])
print(f"Jaccard médio entre folds: {np.mean(jaccards):.6f}")
print(f"interseção dos cinco folds: {len(set.intersection(*selected_sets))}")

O score contaminado não revela um modelo “bom”; revela que o ranking consultou os rótulos das validações. Com seleção dentro do pipeline, o resultado volta à vizinhança do acaso e as listas instáveis denunciam correlações espúrias.

## 4. Contraprova temporal: disponível agora ou depois?

Criamos uma sequência cronológica. A feature segura é o estado observado no passo anterior. `post_outcome` é construída a partir do próprio desfecho e só apareceria depois de \(t_0\). O corte temporal usa as primeiras 1.800 observações para treino e as últimas 600 para teste.

In [ ]:
rng_time = np.random.default_rng(SEED + 1)
n_time = 2_400
latent = np.zeros(n_time)
for t in range(1, n_time):
    latent[t] = 0.92 * latent[t - 1] + rng_time.normal(0, 0.55)

event_prob = 1 / (1 + np.exp(-1.4 * latent))
y_time = rng_time.binomial(1, event_prob)
safe_lag = np.r_[0.0, latent[:-1]]
post_outcome = y_time + rng_time.normal(0, 0.18, n_time)
cut = 1_800

safe_X = safe_lag[:, None]
leaky_time_X = np.column_stack((safe_lag, post_outcome))

def temporal_auc(X):
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2_000, random_state=SEED),
    )
    model.fit(X[:cut], y_time[:cut])
    return roc_auc_score(y_time[cut:], model.predict_proba(X[cut:])[:, 1])

safe_auc = temporal_auc(safe_X)
post_auc = temporal_auc(leaky_time_X)
print(f"somente lag disponível: {safe_auc:.6f}")
print(f"com pós-desfecho:        {post_auc:.6f}")

A métrica quase perfeita não é evidência de implantação possível. Um dicionário de features deve declarar origem, evento, processamento, \(t_0\), janela e política de atualização. A auditoria point-in-time vem antes da comparação de modelos.

## 5. Síntese visual e verificações automáticas

As barras abaixo juntam quatro comparações com perguntas diferentes. Não compare suas alturas como se viessem do mesmo dataset; observe a direção prevista por cada hipótese.

In [ ]:
comparison_names = [
    "hora bruta", "hora cíclica",
    "linear", "interação",
    "seleção vazada", "seleção correta",
    "lag seguro", "pós-desfecho",
]
comparison_values = [
    raw_hour_cv.mean(), cyclic_hour_cv.mean(),
    xor_raw.mean(), xor_poly.mean(),
    leaky_cv.mean(), honest_cv.mean(),
    safe_auc, post_auc,
]
colors = ["#999999", "#2a9d8f"] * 4

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(comparison_names, comparison_values, color=colors)
ax.axhline(0.5, color="black", linestyle="--", linewidth=1)
ax.set(ylabel="ROC-AUC", ylim=(0.4, 1.02), title="Representação útil e vazamentos")
ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()

In [ ]:
assert cyclic_hour_cv.mean() > raw_hour_cv.mean() + 0.25
assert hour_test_auc > 0.90
assert xor_poly.mean() > xor_raw.mean() + 0.25
assert leaky_cv.mean() > honest_cv.mean() + 0.20
assert 0.40 < honest_cv.mean() < 0.65
assert np.mean(jaccards) < 0.25
assert post_auc > safe_auc + 0.15
assert post_auc > 0.98

print("Todas as verificações passaram.")
print({
    "hora_bruta_cv": round(raw_hour_cv.mean(), 6),
    "hora_ciclica_cv": round(cyclic_hour_cv.mean(), 6),
    "hora_ciclica_teste": round(hour_test_auc, 6),
    "xor_linear_cv": round(xor_raw.mean(), 6),
    "xor_interacao_cv": round(xor_poly.mean(), 6),
    "selecao_vazada_cv": round(leaky_cv.mean(), 6),
    "selecao_correta_cv": round(honest_cv.mean(), 6),
    "jaccard_medio": round(float(np.mean(jaccards)), 6),
    "lag_seguro_teste": round(safe_auc, 6),
    "pos_desfecho_teste": round(post_auc, 6),
})

## Conclusões

- A representação cíclica expressou a topologia do relógio e generalizou no teste lacrado.
- A interação tornou visível um sinal que scores univariados quase não detectavam.
- Selecionar antes da CV produziu otimismo forte em dados sem qualquer sinal.
- A baixa estabilidade das listas de ruído reforçou que “feature selecionada” não é sinônimo de variável substantivamente válida.
- A feature pós-desfecho mostrou por que disponibilidade temporal não pode ser inferida pela métrica.

O experimento não prova que seno/cosseno, polinômios ou seleção de 20 features são melhores em geral. Ele demonstra como formular, isolar e testar hipóteses de representação com um protocolo auditável.

## Próximo passo

Na [Aula 20](../aulas/20-interpretabilidade-modelos.md), a pergunta muda de “quais features entram?” para “como o modelo usa as features?”. Coeficientes, permutation importance e SHAP serão tratados sem confundir explicação preditiva com causalidade.